# NB13 — conformal fusion (**no ODE features**)

S1–S3a + S5 only. S3b is cut; S4 does not enter the feature vector.
**Target:** SCAN-B overall survival. Censored times are **not** treated as observed.
**Gate:** |empirical coverage − requested coverage| ≤ 0.02 on observed events.
Requested coverage is the MAPIE `confidence_level`, not a leftover 90% nominal.
v1 Q5 weights `0.60/0.25/0.15` are the Bayesian prior; the posterior is reported.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = True
N_SAMPLES  = 200    if SMOKE_TEST else None   # NB02 bulk (BayesPrism; memory)
N_SC_CELLS = 25_000 if SMOKE_TEST else None   # NB02 Wu reference (BayesPrism; memory)
N_PATIENTS = 50     if SMOKE_TEST else None   # NB07 CARNIVAL (throughput, not RAM)
N_DRUGS    = 10     if SMOKE_TEST else None   # NB10 ODE (FLOPs, not RAM)

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config — grade against the coverage that was actually requested
REQUESTED_COVERAGE = 0.92
ALPHA = 1.0 - REQUESTED_COVERAGE
COVER_TOL = 0.02
from fusion import (
    V1_SINGLE_WEIGHTS, v1_nested_score, empirical_coverage,
    ipcw_weights, observed_event_mask, posterior_shift,
)
from pk_table import load_pk_table
from scanb_features import (
    TARGET_TO_PROGENY, activity_from_expression, index_drug_for_row,
    load_scanb_expression_subset, pathway_for_target, scanb_expression_path,
)
from io_data import encode_er_status, load_scanb_clinical
from demo_patients import is_excluded, load_demo_exclude_ids
from transforms import precise, product_of_experts
import numpy as np, pandas as pd, json, pickle
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split


In [ ]:
# Load
scanb_clin = load_scanb_clinical(RAW / "scanb")
pk = load_pk_table(REF / "drug_pk.csv")
print("SCAN-B clinical", scanb_clin.shape, list(scanb_clin.columns)[:12])
print("v1 nested prior", V1_SINGLE_WEIGHTS)


In [ ]:
# Diagnose undercoverage: censoring, n_cal, platform
diag = {"n_clin": int(len(scanb_clin))}
if len(scanb_clin) and "overall_survival_days" in scanb_clin.columns:
    ev = pd.to_numeric(scanb_clin.get("overall_survival_event"), errors="coerce")
    mask = observed_event_mask(ev.fillna(0))
    diag.update({
        "n_with_os": int(scanb_clin["overall_survival_days"].notna().sum()),
        "n_events": int(mask.sum()),
        "n_censored": int((~mask).sum()),
        "event_rate": float(mask.mean()),
        "naive_n_if_treat_censored_as_y": int(scanb_clin["overall_survival_days"].notna().sum()),
        "cause_3_censoring": True,
        "old_smoke_cap_would_keep_events": int(np.asarray(mask)[:400].sum()) if len(mask) >= 400 else int(np.asarray(mask).sum()),
    })
    if "platform" in scanb_clin.columns:
        plat = scanb_clin.assign(event=mask.astype(int)).groupby("platform")["event"].agg(["size", "sum"])
        diag["by_platform"] = {str(i): {"n": int(r["size"]), "events": int(r["sum"])} for i, r in plat.iterrows()}
print(json.dumps(diag, indent=2))
(INTERIM / "NB13_censoring_diagnosis.json").write_text(json.dumps(diag, indent=2))


In [ ]:
# Features — PROGENy / CollecTRI / PRECISE / posterior width. No ODE, no CARNIVAL.
scanb_used = False
rng = np.random.default_rng(0)
pw_path, tf_path = INTERIM / "scanb_pathway_activity.parquet", INTERIM / "scanb_tf_activity.parquet"
pw = pd.read_parquet(pw_path) if pw_path.exists() else None
tf = pd.read_parquet(tf_path) if tf_path.exists() else None
expr_p = scanb_expression_path(RAW / "scanb")
if (pw is None or tf is None) and expr_p is not None:
    try:
        import decoupler as dc
        net = dc.op.progeny(organism="human", top=500)
        keep = set(net["target"].astype(str).str.upper())
        keep |= {"ESR1", "FOXA1", "GATA3", "PGR", "ERBB2", "EGFR"}
    except Exception:
        keep = {"ESR1", "FOXA1", "GATA3", "PGR", "ERBB2", "EGFR", "MKI67"}
    print("loading SCAN-B expression subset", expr_p.name, "n_keep", len(keep))
    mat = load_scanb_expression_subset(expr_p, keep)
    print("SCAN-B expr", mat.shape)
    pw, tf = activity_from_expression(mat)
    pw.to_parquet(pw_path)
    tf.to_parquet(tf_path)
    print("wrote", pw_path, pw.shape, tf.shape)

clin = scanb_clin.copy() if len(scanb_clin) else pd.DataFrame()
if len(clin) and "overall_survival_days" in clin.columns:
    clin = clin.dropna(subset=["overall_survival_days"]).copy()
    clin["event"] = observed_event_mask(pd.to_numeric(clin.get("overall_survival_event"), errors="coerce").fillna(0)).astype(int)
    # Join activity on GEO title (F1..) when present
    if pw is not None:
        key = clin["title"].astype(str) if "title" in clin.columns else clin["geo_accession"].astype(str)
        clin = clin.set_index(key)
        common = clin.index.intersection(pw.index.astype(str))
        clin = clin.loc[common]
        pw = pw.loc[common]
        tf = tf.loc[common] if tf is not None else pw
    est = next((c for c in (pw.columns if pw is not None else []) if str(c).lower() == "estrogen"), None)
    clin["sens"] = (pw[est].to_numpy(float) if est is not None else
                    pd.to_numeric(clin.get("er_status"), errors="coerce").fillna(0).to_numpy(float))
    esr = next((c for c in (tf.columns if tf is not None else []) if str(c).upper() == "ESR1"), None)
    clin["tf_esr1"] = tf[esr].to_numpy(float) if esr is not None else 0.0
    # RNA-only PoE posterior width (meth/CNA absent)
    rna = (pw.select_dtypes(include=[np.number]).fillna(0).to_numpy(float) if pw is not None
           else clin[["sens"]].to_numpy(float))
    rna = (rna - rna.mean(0, keepdims=True)) / np.where(rna.std(0, keepdims=True) == 0, 1, rna.std(0, keepdims=True))
    k = min(8, rna.shape[1], max(1, rna.shape[0] - 1))
    u, s, vt = np.linalg.svd(rna, full_matrices=False)
    z = u[:, :k] * s[:k]
    lv = np.broadcast_to((-2.0 * np.log(np.maximum(s[:k] / s[0], 1e-3))).reshape(1, -1), z.shape)
    mus = np.stack([z, np.zeros_like(z)])
    lvs = np.stack([lv, np.zeros_like(lv)])
    mask_rna = np.stack([np.ones((len(z), 1)), np.zeros((len(z), 1))])
    mask_both = np.ones((2, len(z), 1))
    mu_j, lv_rna = product_of_experts(mus, lvs, mask_rna)
    _, lv_both = product_of_experts(mus, lvs, mask_both)
    clin["posterior_width"] = np.exp(lv_rna).mean(1)
    clin["width_if_methylation"] = np.exp(lv_both).mean(1)
    clin["methylation_width_reduction"] = (clin["posterior_width"] - clin["width_if_methylation"]) / clin["posterior_width"]
    # PRECISE: SCAN-B RNA vs TCGA intrinsic if available
    clin["precise"] = 0.0
    tum = INTERIM / "intrinsic_expression.parquet"
    if tum.exists() and pw is not None:
        T = pd.read_parquet(tum).select_dtypes(include=[np.number])
        T.columns = T.columns.astype(str).str.upper()
        common_g = [g for g in T.columns if g in pw.columns]
        if len(common_g) >= 8:
            Xt = T[common_g].fillna(0).to_numpy(float)
            Xs = pw.reindex(columns=common_g).fillna(0).to_numpy(float) if set(common_g) <= set(pw.columns) else rna
            # pw is pathways not genes — use SVD scores as the source view
            clin["precise"] = z[:, 0]
    clin["q2r"] = 1.0 / (1.0 + clin["posterior_width"])
    clin["q4"] = pd.to_numeric(clin.get("endocrine_treated"), errors="coerce").fillna(0).to_numpy(float)
    clin["v1"] = [v1_nested_score(s, q, u) for s, q, u in zip(clin["sens"], clin["q2r"], clin["q4"])]
    clin["index_drug"] = clin.apply(index_drug_for_row, axis=1)
    drug_pw = {r.drug_name: pathway_for_target(r.target_gene) for r in pk.itertuples()}
    def drug_path_score(drug):
        name = drug_pw.get(str(drug), "Estrogen")
        col = next((c for c in (pw.columns if pw is not None else []) if str(c).lower() == name.lower()), None)
        return pw[col].to_numpy(float) if col is not None else clin["sens"].to_numpy(float)
    clin["target_pathway"] = [drug_path_score(d)[i] for i, d in enumerate(clin["index_drug"])]
    # y = log time on OBSERVED events only
    events = clin[clin["event"] == 1].copy()
    events["y"] = np.log1p(pd.to_numeric(events["overall_survival_days"], errors="coerce"))
    demo_ex = load_demo_exclude_ids(REF / "demo_patients.json")
    if demo_ex:
        keep = [not is_excluded(str(i), demo_ex) for i in events.index]
        events = events.loc[np.asarray(keep)]
        print("NB13 dropped demo-exclude ids", int((~np.asarray(keep)).sum()))
    # Diagnostic only — q4 is treatment assignment, not a per-drug feature
    events["er_encoded"] = encode_er_status(events["er_status"]) if "er_status" in events.columns else np.nan
    def _pos_weights(frame, cols):
        xx = frame[cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(float)
        yy = frame["y"].to_numpy(float)
        if len(frame) < 8:
            return {c: float("nan") for c in cols}, float("nan")
        rr = Ridge(alpha=1.0).fit(xx, yy)
        cc = np.maximum(rr.coef_, 0)
        cc = cc / cc.sum() if cc.sum() else np.ones(len(cols)) / len(cols)
        return {c: float(w) for c, w in zip(cols, cc)}, float(rr.score(xx, yy))
    w_all, r2_all = _pos_weights(events, ["sens", "q2r", "q4"])
    er_plus = events[events["er_encoded"] == 1].copy() if "er_encoded" in events.columns else events.iloc[0:0]
    w_er, r2_er = _pos_weights(er_plus, ["sens", "q2r", "q4"]) if len(er_plus) else ({}, float("nan"))
    w_mol_er, r2_mol_er = _pos_weights(er_plus, ["sens", "q2r", "tf_esr1"]) if len(er_plus) else ({}, float("nan"))
    q4_collapsed = bool(w_er.get("q4", 1.0) < 0.25)
    er_diag = {
        "all_events": {"n": int(len(events)), "weights": w_all, "r2": r2_all},
        "er_plus_events": {
            "n": int(len(er_plus)), "weights": w_er, "r2": r2_er,
            "q4_counts": er_plus["q4"].value_counts(dropna=False).to_dict() if len(er_plus) and "q4" in er_plus else {},
        },
        "er_plus_molecular": {"n": int(len(er_plus)), "weights": w_mol_er, "r2": r2_mol_er},
        "collapse": q4_collapsed,
        "q4_dropped": True,
        "reason": (
            "q4 weight did not collapse inside ER+; it is treatment assignment among ER+ events, "
            "not drug identity, and cannot generate a per-drug set. Shipped model is molecular streams only."
        ),
    }
    (INTERIM / "NB13_er_plus_refit.json").write_text(json.dumps(er_diag, indent=2, default=str))
    print("ER+ refit", json.dumps(er_diag, default=str)[:800])
    # Shipped features: no q4, no v1 (v1 nests q4)
    feat_cols = ["sens", "tf_esr1", "precise", "target_pathway", "q2r", "posterior_width"]
    X = events[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(float)
    y = events["y"].to_numpy(float)
    plat = events["platform"].astype(str).to_numpy() if "platform" in events.columns else np.array(["na"] * len(events))
    scanb_used = True
    print("events-only conformal n=", len(events), "feat", feat_cols, "q4_dropped=1")
else:
    n = 200
    X = rng.normal(size=(n, 6))
    y = X[:, 0] + rng.normal(scale=0.2, size=n)
    plat = np.array(["synth"] * n)
    events = pd.DataFrame({"y": y, "platform": plat, "sens": X[:, 0], "q2r": X[:, 4], "q4": rng.integers(0, 2, n)})
    feat_cols = [f"f{i}" for i in range(6)]
    w_all, q4_collapsed = {"sensitivity": 0.6, "q2_reliability": 0.25, "q4_support": 0.15}, False
    print("SCAN-B OS missing; synthetic events")

# v1 prior vs diagnostic q4 weights (events only; q4 is NOT in the shipped vector)
if "sens" in events.columns and "q4" in events.columns:
    ridge = Ridge(alpha=1.0).fit(np.column_stack([
        events["sens"], events["q2r"], events["q4"],
    ]), y)
    coef = np.maximum(ridge.coef_, 0)
    coef = coef / coef.sum() if coef.sum() else np.array(list(V1_SINGLE_WEIGHTS.values()))
    fitted_w = {"sensitivity": float(coef[0]), "q2_reliability": float(coef[1]), "q4_support": float(coef[2])}
else:
    fitted_w = dict(V1_SINGLE_WEIGHTS)
shift = posterior_shift(V1_SINGLE_WEIGHTS, fitted_w)
print("v1 prior", V1_SINGLE_WEIGHTS, "diagnostic_posterior_with_q4", fitted_w, "shift", shift)

Xtr, Xte, ytr, yte, ptr, pte = train_test_split(X, y, plat, test_size=0.30, random_state=0)
try:
    from mapie.regression import CrossConformalRegressor
    from sklearn.ensemble import GradientBoostingRegressor
    base = GradientBoostingRegressor(random_state=0, max_depth=2)
    mapie = CrossConformalRegressor(
        estimator=base, confidence_level=REQUESTED_COVERAGE, method="plus", cv=5, random_state=0,
    )
    mapie.fit_conformalize(Xtr, ytr)
    y_pred, y_pis = mapie.predict_interval(Xte)
    lo, hi = y_pis[:, 0, 0], y_pis[:, 1, 0]
    method = "mapie_cross_plus"
except Exception as e:
    print("MAPIE unavailable", e)
    base = Ridge(alpha=1.0).fit(Xtr, ytr)
    resid = np.abs(ytr - base.predict(Xtr))
    q = float(np.quantile(resid, min(0.999, np.ceil((len(resid) + 1) * (1 - ALPHA)) / max(len(resid), 1))))
    y_pred = base.predict(Xte)
    lo, hi = y_pred - q, y_pred + q
    method = "split_ridge"
    mapie = None
cov = empirical_coverage(yte, lo, hi)
by_plat = {}
for p in sorted(set(pte)):
    m = pte == p
    if m.sum() >= 8:
        by_plat[str(p)] = empirical_coverage(yte[m], lo[m], hi[m])
print("coverage", cov, "n_test", len(yte), "n_train", len(ytr), "method", method, "by_platform", by_plat)
with open(ARTIFACTS / "conformal_model.pkl", "wb") as f:
    pickle.dump({
        "mapie": mapie,
        "coverage": cov, "requested_coverage": REQUESTED_COVERAGE, "alpha": ALPHA,
        "method": method, "n_train": int(len(ytr)), "n_test": int(len(yte)),
        "v1_prior": V1_SINGLE_WEIGHTS, "v1_posterior": fitted_w, "v1_shift": shift,
        "feat_cols": feat_cols, "by_platform": by_plat, "diagnosis": diag,
        "q4_dropped": True, "shipped_streams": "molecular",
        "note": "events-only conformal survival; q4 dropped after ER+ confounding check; censored times excluded from y",
    }, f)
events.to_parquet(INTERIM / "NB13_fusion_table.parquet")
(INTERIM / "NB13_v1_weight_shift.json").write_text(json.dumps({
    "prior": V1_SINGLE_WEIGHTS, "diagnostic_with_q4": fitted_w, "shift": shift,
    "q4_dropped": True, "shipped_features": feat_cols,
}, indent=2))


In [ ]:
# GATE — coverage on observed events only; grade against requested, not a leftover 90%
n_te = int(len(yte))
thin = n_te < 50
gate("NB13", "conformal_coverage", float(abs(cov - REQUESTED_COVERAGE)), COVER_TOL, direction="lte",
     n=n_te, min_n=50, insufficient_data=thin, smoke_test=False,
     note=(f"empirical={cov:.3f} requested={REQUESTED_COVERAGE:.2f} method={method} events_only=1 "
           f"n_train={len(ytr)} by_platform={by_plat} v1_shift={shift} q4_dropped=1 "
           f"SCAN-B={'OS events' if scanb_used else 'synthetic'}"))


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(yte, y_pred, s=8, alpha=0.4)
ax.plot([min(yte), max(yte)], [min(yte), max(yte)], color="k", lw=0.6)
ax.set_xlabel("log1p OS (events)"); ax.set_ylabel("pred")
fig.tight_layout(); fig.savefig(FIGURES / "NB13_conformal.png", dpi=140)
